In [1]:
import sleap
import numpy as np
import matplotlib.pyplot as plt
import os, sys
import datetime

sys.path.append('..')
from python.postprocess import *
from python.animation import *
from autoencoder.src.data_loader import *
from autoencoder.min2.run_models import *

from tensorflow.keras.datasets import fashion_mnist

ModuleNotFoundError: No module named 'polygon_based_correction'

In [4]:
(x_train, _), (x_test, _) = fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print (x_train.shape)
print (x_test.shape)

4431872/4422102 [==============================] - 0s 0us/step
(60000, 28, 28)
(10000, 28, 28)


In [11]:
def get_ae_model_1(input_shape=(28, 28), latent_dim=32, num_layers=4):
    # simple feedforward autoencoder
    inputs = Input(shape=input_shape)
    layer_dim = latent_dim * (2 ** num_layers)
    
    x = Flatten()(inputs)
    # encoder
    for _ in range(num_layers):
        x = Dense(layer_dim, activation='relu')(x)
        layer_dim //= 2
    x = Dense(latent_dim, activation='relu')(x)
    
    # decoder
    for _ in range(num_layers):
        x = Dense(layer_dim, activation='relu')(x)
        layer_dim *= 2
    
    outputs = Dense(input_shape[0] * input_shape[1], activation='sigmoid')(x)
    outputs = Reshape(input_shape)(outputs)

    model = Model(inputs, outputs)
    model.summary()
    
    return model

In [12]:
m1 = get_ae_model_1(latent_dim=64, num_layers=2)
m1.compile(optimizer='adam', loss='mse')

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 28, 28)]          0         
                                                                 
 flatten_2 (Flatten)         (None, 784)               0         
                                                                 
 dense_18 (Dense)            (None, 256)               200960    
                                                                 
 dense_19 (Dense)            (None, 128)               32896     
                                                                 
 dense_20 (Dense)            (None, 64)                8256      
                                                                 
 dense_21 (Dense)            (None, 64)                4160      
                                                                 
 dense_22 (Dense)            (None, 128)               8320

In [14]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1)

In [15]:
m1.fit(x_train, x_train,
                epochs=100,
                shuffle=True,
                validation_data=(x_test, x_test), 
                callbacks=[tensorboard_callback, early_stopping_callback])

Epoch 1/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0105 - val_loss: 0.0104
Epoch 2/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0102 - val_loss: 0.0101
Epoch 3/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0100 - val_loss: 0.0100
Epoch 4/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0097 - val_loss: 0.0099
Epoch 5/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0095 - val_loss: 0.0096
Epoch 6/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0094 - val_loss: 0.0095
Epoch 7/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0092 - val_loss: 0.0093
Epoch 8/100
1875/1875 [==============================] - 9s 5ms/step - loss: 0.0091 - val_loss: 0.0093
Epoch 9/100
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0090 - val_loss: 0.0094
Epoch 10/100
1875/1875 [==============================] - 9s 5ms/